In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
from google.colab import drive
import os  

drive.mount('/content/drive')

# --- Define Project Paths ---

# Base folder for project in Google Drive
BASE_PROJECT_PATH = r"/content/drive/MyDrive/Nifty50_RL_Project"

# Define all the sub-folders
PROCESSED_DIR = os.path.join(BASE_PROJECT_PATH, "finance_data", "processed")
TICKER_LIST_PATH = os.path.join(BASE_PROJECT_PATH, "nifty50.csv") # Path to ticker file

# Create the processed directory if it doesn't exist
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Paths are set. Processed data will save to: {PROCESSED_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Paths are set. Processed data will save to: /content/drive/MyDrive/Nifty50_RL_Project/finance_data/processed


In [ ]:
import pandas as pd
import yfinance as yf
import datetime

print(f"Loading tickers from: {TICKER_LIST_PATH}")
try:
    tickers_df = pd.read_csv(TICKER_LIST_PATH)
    nifty_50_tickers = tickers_df['ticker'].tolist()
    print(f"Found {len(nifty_50_tickers)} tickers.")
except FileNotFoundError:
    print(f"Error: Could not find 'nifty50.csv' at {TICKER_LIST_PATH}")
    # Script stops if file not found
except Exception as e:
    print(f"An error occurred: {e}")

# Set the start and end dates
start_date = "2007-01-01"
end_date = datetime.date.today().strftime('%Y-%m-%d')

print(f"Downloading data for {len(nifty_50_tickers)} tickers from {start_date} to {end_date}...")

try:
    # 1. Download all ticker data at once
    data = yf.download(nifty_50_tickers, start=start_date, end=end_date)
    print("Download complete.")

    # 2. Re-format from "wide" to "long"
    print("Reformatting data for FinRL...")
    df = data.stack(level=1, future_stack=True).rename_axis(['Date', 'Ticker']).reset_index()

    # 3. Clean and format columns for FinRL
    df = df.rename(columns={
        'Date': 'date',
        'Ticker': 'tic',
        'Open': 'open',
        'High': 'high',
        'Low': 'low',
        'Close': 'close',
        'Volume': 'volume'
    })

    # Columns FinRL needs
    df = df[['date', 'tic', 'open', 'high', 'low', 'close', 'volume']]

    # 4. Sort data (CRITICAL for time-series)
    df = df.sort_values(by=['date', 'tic']).reset_index(drop=True)

    # 5. Check for missing values (NaNs)
    initial_rows = len(df)
    df = df.dropna()
    final_rows = len(df)

    if initial_rows > final_rows:
        print(f"Dropped {initial_rows - final_rows} rows with missing values (NaNs).")

    print("\n--- Data is clean and in 'long' format! ---")

    # 6. SAVE to Google Drive
    clean_filepath = os.path.join(PROCESSED_DIR, "_NIFTY50_CLEAN_LONG_FORMAT.csv")
    df.to_csv(clean_filepath, index=False)
    print(f"✅ Successfully saved clean data to: {clean_filepath}")

    # 7. Verify the final DataFrame
    print("\n--- Final DataFrame Head ---")
    print(df.head())
    print("\n--- Final DataFrame Info ---")
    df.info()

except Exception as e:
    print(f"An error occurred during download/processing: {e}")

Loading tickers from: /content/drive/MyDrive/Nifty50_RL_Project/nifty50.csv
Found 50 tickers.


/tmp/ipython-input-1869472793.py:25: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(nifty_50_tickers, start=start_date, end=end_date)
[*********************100%***********************]  50 of 50 completed


Download complete.
Reformatting data for FinRL...
Dropped 9091 rows with missing values (NaNs).

--- Data is clean and in 'long' format! ---
✅ Successfully saved clean data to: /content/drive/MyDrive/Nifty50_RL_Project/finance_data/processed/_NIFTY50_CLEAN_LONG_FORMAT.csv

--- Final DataFrame Head ---
Price       date            tic        open        high         low  \
0     2007-01-02    ADANIENT.NS   14.129113   14.451626   14.036967   
2     2007-01-02  APOLLOHOSP.NS  195.488504  196.364837  188.747521   
3     2007-01-02  ASIANPAINT.NS   61.665444   62.079863   61.085255   
4     2007-01-02    AXISBANK.NS   83.338490   83.338490   81.483777   
5     2007-01-02  BAJAJ-AUTO.NS  414.745256  434.083803  412.138091   

Price       close     volume  
0       14.236618  2117578.0  
2      189.983368   126540.0  
3       61.383636   139410.0  
4       82.649597   907130.0  
5      429.370544   460494.0  

--- Final DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
Index: 222859 en